## **Partie 1 : Charger les documents et exécuter le modèle de réorganisation**

### **Installation de pinecone**

In [2]:
%pip install -U pinecone pinecone-notebooks --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 16.1 MB/s eta 0:00:00


### **Authentifiez-vous avec une pomme de pin**

In [3]:
import os
if not os.environ.get("pcsk_671uUm_HLpaeFgfmLcDaiPhJJjRxJCSx86GFX4fc4p6UvCQJU4pEkvPz3hhnSuxT83EGfF"):
    from pinecone_notebooks.colab import Authenticate
    Authenticate()

### **Instancier le client Pinecone**

In [4]:
from pinecone import Pinecone
api_key = os.environ.get("pcsk_671uUm_HLpaeFgfmLcDaiPhJJjRxJCSx86GFX4fc4p6UvCQJU4pEkvPz3hhnSuxT83EGfF")
pc = Pinecone(api_key=api_key)

### **Définissez votre requête et vos documents**

In [28]:
query = "Ecosystème apple"
documents = [
    "Les pommes sont des fruits croquants et sucrés qui poussent sur les arbres. Il en existe de nombreuses variétés comme la Gala, la Fuji et la Granny Smith.", # Ajouter un document sur les pommes
    "Apple Inc. conçoit, fabrique et commercialise des smartphones, des ordinateurs personnels, des tablettes, des objets connectés et des accessoires dans le monde entier. Ses produits les plus célèbres sont l'iPhone, le Mac, l'iPad et l'Apple Watch.", # Ajouter un document sur les produits Apple
    "Une pomme mûre se mange souvent crue, mais elle peut aussi être utilisée dans les tartes, les compotes et les cidres. Elle est une bonne source de fibres et de vitamine C.", # Ajouter un autre document sur les fruits
    "L'App Store d'Apple propose des millions d'applications pour ses appareils, offrant un vaste écosystème de logiciels et de services.", # Ajouter un autre document sur l'entreprise
    "Parmi les services Apple populaires, on trouve Apple Music, Apple TV+, Apple Arcade et iCloud, qui proposent des abonnements pour le divertissement et le stockage dans le nuage." # Ajouter un autre document (au choix)
]

### **Appelez le service de réévaluation**

In [7]:
from pinecone import RerankModel

In [29]:
reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n = 3
)

### **Examiner les résultats reclassés**

In [30]:
def show_reranked_results(query, matches):
    print(f"Query: {query}")
    for i, m in enumerate(matches):
        # Print the position (i+1), m.score, and m.document.text
        print(f"{i+1}. Score: {m.score}, Text: {m.document['text']}")

show_reranked_results(query, reranked.data) # Fill in the correct attribute

Query: Ecosystème apple
1. Score: 0.9623913, Text: L'App Store d'Apple propose des millions d'applications pour ses appareils, offrant un vaste écosystème de logiciels et de services.
2. Score: 0.022933563, Text: Apple Inc. conçoit, fabrique et commercialise des smartphones, des ordinateurs personnels, des tablettes, des objets connectés et des accessoires dans le monde entier. Ses produits les plus célèbres sont l'iPhone, le Mac, l'iPad et l'Apple Watch.
3. Score: 0.0030278352, Text: Parmi les services Apple populaires, on trouve Apple Music, Apple TV+, Apple Arcade et iCloud, qui proposent des abonnements pour le divertissement et le stockage dans le nuage.


## **Partie 2 : Mise en place d’un index sans serveur pour les notes médicales**

### **Installez les bibliothèques de données et de modèles**

In [31]:
%pip install pandas torch transformers --quiet

### **Importer les modules et définir les paramètres d'environnement**

In [32]:
import os
import time
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch

In [33]:
# Get cloud and region settings (these are defaults that work for most users)
cloud = os.getenv('PINECONE_CLOUD', 'aws') # e.g., 'aws'
region = os.getenv('PINECONE_REGION', 'us-east-1') # e.g., 'us-east-1'

# Define serverless specifications
spec = ServerlessSpec(cloud=cloud, region=region)

# Define index name
index_name = 'my-agent' # Give your index a name

### **Créer ou recréer l'index**

In [35]:
# Clean up any existing index with the same name
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

# Create a new index
pc.create_index(
    name=index_name,
    # This matches our embedding model size
    dimension=384,
    # Distance metric for similarity
    metric = "cosine",
    spec=spec
)

IndexModel(name='my-agent', metric='cosine', status=IndexStatus(ready=True, state='Ready'), spec=IndexSpec(serverless=ServerlessSpecInfo(cloud='aws', region='us-east-1', read_capacity={'mode': 'OnDemand', 'status': {'state': 'Ready', 'current_shards': None, 'current_replicas': None}}, source_collection=None, schema=None), pod=None, byoc=None), host='https://my-agent-b6glew5.svc.aped-4627-b74a.pinecone.io', private_host=None, vector_type='dense', dimension=384, deletion_protection='disabled', tags=None, embed=None, created_at=None)

## **Partie 3 : Charger les données d’exemple**

### **Téléchargement et lecture du fichier JSONL**

In [36]:
import requests
import tempfile

In [39]:

with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")

    # Download the file from github
    url = "https://raw.githubusercontent.com/pinecone-io/examples/main/docs/data/sample_notes_data.jsonl" # Insert the GitHub raw URL here
    response = requests.get(url)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

    df = pd.read_json(file_path, orient='records', lines=True)

### **Prévisualisation du DataFrame**

In [40]:
# Show head of the DataFrame
print("Data shape:", df.shape) # Show number of rows and columns
df.head()

Data shape: (100, 3)


,id,values,metadata
0,P011,"[-0.2027486265, 0.2769146562, -0.1509393603, 0...","{'advice': 'rest, hydrate', 'symptoms': 'heada..."
1,P001,"[0.1842793673, 0.4459365904, -0.0770567134, 0....","{'tests': 'EKG, stress test', 'symptoms': 'che..."
2,P002,"[-0.2040648609, -0.1739618927, -0.2897160649, ...","{'HbA1c': '7.2', 'condition': 'diabetes', 'med..."
3,P003,"[0.1889383644, 0.2924542725, -0.2335938066, -0...","{'symptoms': 'cough, wheezing', 'diagnosis': '..."
4,P004,"[-0.12171068040000001, 0.1674752235, -0.231888...","{'referral': 'dermatology', 'condition': 'susp..."


## **Partie 4 : Mise à jour des données dans l’index**

### **Instanciation du client d'index et mise à jour**

In [41]:
# Instantiate an index client
index = pc.Index(name=index_name)

# Upsert data into index from DataFrame
index.upsert_from_dataframe(df) # Pass the DataFrame

Upserting:   0%|          | 0/1 [00:00<?, ?batch/s]

UpsertResponse(SUCCESS: 100/100 items, 1/1 batches)

### **Attendons la disponibilité**

In [43]:
def is_fresh(index):
    stats = index.describe_index_stats()
    vector_count = stats.total_vector_count
    print(f"Vector count: ", vector_count)
    return vector_count >= len(df) # What should this be?

while not is_fresh(index):
    time.sleep(5)

print("Index ready!")
index.describe_index_stats()

Vector count:  100
Index ready!


DescribeIndexStatsResponse(dimension=384, total_vector_count=100, metric='cosine', namespaces=1)

## **Partie 5 : Fonction de requête et d’intégration**

### **Définissons notre fonction d'intégration**

In [48]:
def get_embedding(input_question):
  model_name = 'sentence-transformers/all-MiniLM-L6-v2'
  tokenizer = AutoTokenizer.from_pretrained(model_name)
  model = AutoModel.from_pretrained(model_name)
  encoded_input = tokenizer(input_question, padding=True, truncation=True, return_tensors='pt')
  with torch.no_grad():
    model_output = model(**encoded_input)
    embedding = model_output.last_hidden_state[0].mean(dim=0) # Which dimension to average?
    return embedding

### **Exécutons une requête de recherche sémantique**

In [50]:
# Build a query to search
question = "I'm very sick, what traitment can i take" # Ask a medical question
query = get_embedding(question).tolist()

# Get results
results = index.query(vector=[query], top_k=5, include_metadata=True)

# Sort results by score in descending order
sorted_matches = sorted(results['matches'], key=lambda x: x['score'], reverse=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## **Partie 6 : Afficher et réorganiser les notes cliniques**

### **Affichons les premiers résultats de la recherche**

In [52]:
def show_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nResults:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match["id"]}')
        print(f' Score: {match["score"]}') # What field contains the score?
        print(f' Metadata: {match["metadata"]}') # What field contains metadata?
        print('')

show_results(question, sorted_matches)

Question: 'I'm very sick, what traitment can i take'

Results:
   1. ID: P005
 Score: 0.391939729
 Metadata: {'preventive_care': 'flu vaccination', 'vital_signs': 'normal'}

   2. ID: P051
 Score: 0.380865693
 Metadata: {'diagnosis': 'flu', 'symptoms': 'fever, body aches', 'treatment': 'symptomatic'}

   3. ID: P099
 Score: 0.380865693
 Metadata: {'diagnosis': 'flu', 'symptoms': 'fever, body aches', 'treatment': 'symptomatic'}

   4. ID: P089
 Score: 0.341233313
 Metadata: {'advice': 'monitor blood pressure', 'vital_signs': 'slightly elevated'}

   5. ID: P041
 Score: 0.341233313
 Metadata: {'advice': 'monitor blood pressure', 'vital_signs': 'slightly elevated'}



### **Préparons les documents pour le réexamen**

In [54]:
# Create documents with concatenated metadata field as "reranking_field" field
transformed_documents = [
    {
        'id': match['id'],
        'reranking_field': '; '.join([f"{key}: {value}" for key, value in match['metadata'].items()])
    }
    for match in results['matches']
]

### **Exécutons le réordonnancement sans serveur**

In [55]:
# Define a more specific query for reranking
refined_query = "My heart is beak" # Make a more specific medical question

# Perform reranking based on the query and specified field
reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=5, # How many top results do you want?
    return_documents=True,
)

## **Affichons les résultats reclassés**

In [57]:
def show_reranked_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nReranked Results:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match['document']['id']}')
        print(f' Score: {match['score']}')
        print(f' Reranking Field: {match['document']['reranking_field']}')
        print('')
show_reranked_results(refined_query, reranked_results.data)

Question: 'My heart is beak'

Reranked Results:
   1. ID: P089
 Score: 0.0016166499
 Reranking Field: advice: monitor blood pressure; vital_signs: slightly elevated

   2. ID: P041
 Score: 0.0016166499
 Reranking Field: advice: monitor blood pressure; vital_signs: slightly elevated

   3. ID: P051
 Score: 1.8342893e-05
 Reranking Field: diagnosis: flu; symptoms: fever, body aches; treatment: symptomatic

   4. ID: P099
 Score: 1.8342893e-05
 Reranking Field: diagnosis: flu; symptoms: fever, body aches; treatment: symptomatic

   5. ID: P005
 Score: 1.5936621e-05
 Reranking Field: preventive_care: flu vaccination; vital_signs: normal



### **Nettoyage**

In [58]:
# Delete the index to save resources
pc.delete_index(name=index_name)